# Fine-tune Gemma 4 E2B on three insurance datasets

This Colab notebook fine-tunes **one text-only `unsloth/gemma-4-E2B-it` model** with Unsloth QLoRA on a combined dataset made from:

- [InsuranceQA V2](https://github.com/shuzi/insuranceQA/tree/master/V2): uses the official train file, decodes its `idx_*` tokens, and pairs each question with its first available ground-truth answer.
- [Bitext Insurance Chatbot Dataset](https://huggingface.co/datasets/bitext/Bitext-insurance-llm-chatbot-training-dataset): uses its `instruction`, `intent`, `category`, `tags`, and `response` fields.
- [Insurance AI Reliability Benchmark](https://huggingface.co/datasets/pashas/insurance-ai-reliability-benchmark): uses **train only**. Validation and test are deliberately not used for training.

The notebook turns these into three clearly prompted record-generation tasks, mixes them into one SFT dataset, and saves a single LoRA adapter. At generation time it samples empirical labels (domain, intent/category/tags, and benchmark category/difficulty/routing) from the training records before asking Gemma to invent the remaining content. This makes categorical proportions much more controllable than an unconstrained “generate data” prompt. It does not guarantee exact matching of every natural-language or numeric distribution.

**Colab:** choose a GPU runtime (Runtime → Change runtime type → GPU). Gemma weights require Hugging Face access if the model repository prompts for authentication; use the optional login cell only if needed. The InsuranceQA repository page does not clearly present a dataset license, so check its current terms before redistribution or commercial use. Do not use generated examples as real insurance advice or real customer records.

The training sample defaults to 1,500 InsuranceQA rows, 1,500 Bitext rows, and 500 reliability rows. The latter is sampled with replacement from the benchmark's 357-row training split, so some benchmark records repeat. Change the counts below if needed.

## 1. Install the text-training libraries

In [6]:
%%capture
# Correcting the installation dependencies. datasets==4.3.0 is a invalid version target.
!pip install -q sentencepiece protobuf "datasets>=2.18.0" "huggingface_hub>=1.5.0,<2.0" hf_transfer
!pip install -q --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
!pip install -q --no-deps --upgrade "transformers==5.5.0" "tokenizers>=0.22.0,<=0.23.0"

## 2. Check that Colab assigned a GPU

In [2]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("A GPU is required. In Colab choose Runtime > Change runtime type > GPU, then rerun the notebook.")

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

GPU: Tesla T4
PyTorch: 2.11.0+cu130
CUDA available: True


## 3. Imports, reproducibility, and output folder

In [3]:
import os
import re
import io
import gzip
import json
import math
import random
import textwrap
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from datasets import load_dataset, Dataset
from google.colab import drive
from huggingface_hub import notebook_login
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTTrainer, SFTConfig

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Mount Drive so the adapter and generated files survive a Colab session ending.
drive.mount("/content/drive")
OUTPUT_DIR = "/content/drive/MyDrive/gemma_insurance_synthetic"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Outputs will be saved to:", OUTPUT_DIR)

# Only uncomment if Hugging Face asks for authentication to download the base model.
# notebook_login()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Mounted at /content/drive
Outputs will be saved to: /content/drive/MyDrive/gemma_insurance_synthetic


## 4. Download and decode InsuranceQA V2

In [4]:
INSURANCEQA_BASE = "https://raw.githubusercontent.com/shuzi/insuranceqa/master/V2/"
VOCAB_URL = INSURANCEQA_BASE + "vocabulary"
ANSWER_URL = INSURANCEQA_BASE + "InsuranceQA.label2answer.raw.encoded.gz"
QA_TRAIN_URL = INSURANCEQA_BASE + "InsuranceQA.question.anslabel.raw.100.pool.solr.train.encoded.gz"


def download_bytes(url):
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    return response.content


def decode_gzip_url(url):
    return gzip.decompress(download_bytes(url)).decode("utf-8", errors="replace")


# Vocabulary maps each encoded token (idx_...) to readable text.
vocab_text = download_bytes(VOCAB_URL).decode("utf-8", errors="replace")
vocab = {}
for line in vocab_text.splitlines():
    parts = line.split("\t", 1)
    if len(parts) == 2:
        vocab[parts[0].strip()] = parts[1].strip()


def decode_idx_sequence(token_text):
    words = [vocab.get(tok, tok) for tok in token_text.split()]
    text = " ".join(words)
    # Repair common punctuation spacing introduced by joining the vocabulary tokens.
    text = re.sub(r"\s+([,.;:!?%])", r"\1", text)
    text = re.sub(r"([([{])\s+", r"\1", text)
    text = re.sub(r"\s+([)\]}])", r"\1", text)
    text = re.sub(r"\s+(['’])", r"\1", text)
    return re.sub(r"\s+", " ", text).strip()


# Answer IDs start each record at the beginning of a line. Splitting on that
# boundary also remains safe if an answer ever contains a line break.
answer_text = decode_gzip_url(ANSWER_URL)
answer_headers = list(re.finditer(r"(?m)^(\d+)\t", answer_text))
answer_by_id = {}
for i, match in enumerate(answer_headers):
    end = answer_headers[i + 1].start() if i + 1 < len(answer_headers) else len(answer_text)
    encoded_answer = answer_text[match.end():end].strip()
    answer_by_id[match.group(1)] = decode_idx_sequence(encoded_answer)

# This is the official 100-candidate-pool TRAIN split. The retrieval pool in
# column 4 is ignored; column 3 contains ground-truth answer IDs.
qa_train_text = decode_gzip_url(QA_TRAIN_URL)
insuranceqa_rows = []
for line in qa_train_text.splitlines():
    fields = line.split("\t")
    if len(fields) < 3:
        continue
    domain, encoded_question, ground_truth_ids = fields[:3]
    answer = next(
        (answer_by_id[label] for label in ground_truth_ids.split() if label in answer_by_id),
        None,
    )
    if not answer:
        continue
    question = decode_idx_sequence(encoded_question)
    if question and domain.strip():
        insuranceqa_rows.append({
            "domain": domain.strip(),
            "question": question,
            "answer": answer,
        })

if not insuranceqa_rows:
    raise RuntimeError("No InsuranceQA training pairs were decoded. Check the download URLs and file format.")

print("InsuranceQA decoded train pairs:", len(insuranceqa_rows))
print(json.dumps(insuranceqa_rows[0], ensure_ascii=False, indent=2)[:1200])

InsuranceQA decoded train pairs: 12889
{
  "domain": "disability-insurance",
  "question": "Is Disability Insurance Required By Law?",
  "answer": "Not generally. There are five states that require most all employers carry short term disability insurance on their employees. These states are: California, Hawaii, New Jersey, New York, and Rhode Island. Besides this mandatory short term disability law, there is no other legislative imperative for someone to purchase or be covered by disability insurance."
}


## 5. Load Bitext and the benchmark training split

In [10]:
from huggingface_hub import hf_hub_download, list_repo_files
import json
import pandas as pd

BITEXT_ID = "bitext/Bitext-insurance-llm-chatbot-training-dataset"
RELIABILITY_ID = "pashas/insurance-ai-reliability-benchmark"

# 1. Load the Bitext dataset
bitext_ds = load_dataset(BITEXT_ID, split="train")

# 2. Load the Reliability benchmark dataset with a robust fallback
try:
    reliability_ds = load_dataset(RELIABILITY_ID, split="train")
except Exception:
    print("Standard load failed. Inspecting repository files...")
    files = list_repo_files(repo_id=RELIABILITY_ID, repo_type="dataset")
    print("Files found in repo:", files)

    # Try to find any training data file (json, jsonl, csv, or parquet)
    train_file = next((f for f in files if "train" in f.lower() and (f.endswith(".json") or f.endswith(".jsonl") or f.endswith(".parquet") or f.endswith(".csv"))), None)
    if not train_file:
        train_file = next((f for f in files if f.endswith(".json") or f.endswith(".jsonl") or f.endswith(".parquet")), None)

    if not train_file:
        raise RuntimeError("Could not locate a supported dataset file in the repository.")

    print(f"Downloading {train_file} from Hugging Face hub...")
    local_file = hf_hub_download(repo_id=RELIABILITY_ID, repo_type="dataset", filename=train_file)

    if train_file.endswith(".parquet"):
        df = pd.read_parquet(local_file)
        reliability_ds = Dataset.from_pandas(df)
    elif train_file.endswith(".jsonl"):
        reliability_ds = load_dataset("json", data_files=local_file, split="train")
    elif train_file.endswith(".csv"):
        df = pd.read_csv(local_file)
        reliability_ds = Dataset.from_pandas(df)
    else:
        with open(local_file, "r", encoding="utf-8") as f:
            raw_data = json.load(f)
        reliability_ds = Dataset.from_list(raw_data)

# 3. Clean and validate Bitext dataset
BITEXT_FIELDS = ["instruction", "intent", "category", "tags", "response"]
missing_bitext = [name for name in BITEXT_FIELDS if name not in bitext_ds.column_names]
if missing_bitext:
    raise ValueError(f"Bitext columns changed; missing {missing_bitext}. Found: {bitext_ds.column_names}")

bitext_rows = []
for row in bitext_ds:
    clean = {name: str(row.get(name, "") or "").strip() for name in BITEXT_FIELDS}
    if all(clean.values()):
        bitext_rows.append(clean)

# 4. Validate Reliability dataset
reliability_rows = [dict(row) for row in reliability_ds]
required_reliability = {"category", "subcategory", "difficulty", "input", "expected_output", "evaluation_criteria", "metadata"}
if not reliability_rows:
    raise RuntimeError("The reliability benchmark train split loaded empty.")
if not required_reliability.issubset(reliability_rows[0]):
    raise ValueError(f"Reliability schema changed. Found keys: {list(reliability_rows[0])}")

print("Bitext clean train rows:", len(bitext_rows))
print("Reliability benchmark train rows:", len(reliability_rows))
print("Bitext source columns:", bitext_ds.column_names)
print("Reliability example keys:", list(reliability_rows[0].keys()))

Standard load failed. Inspecting repository files...
Files found in repo: ['.gitattributes', 'README.md', 'data/benchmark_v1.jsonl', 'data/sample_predictions.jsonl', 'data/sample_results.json', 'data/test.jsonl', 'data/train.jsonl', 'data/validation.jsonl', 'evaluation/evaluate.py', 'evaluation/example_evaluation.py', 'generate_benchmark.py']
Bitext clean train rows: 39000
Reliability benchmark train rows: 357
Bitext source columns: ['instruction', 'intent', 'category', 'tags', 'response']
Reliability example keys: ['id', 'category', 'subcategory', 'difficulty', 'input', 'expected_output', 'evaluation_criteria', 'metadata']


In [11]:
# Running the loading script to load datasets with our updated dependencies
from huggingface_hub import hf_hub_download, list_repo_files
import json
import pandas as pd

BITEXT_ID = "bitext/Bitext-insurance-llm-chatbot-training-dataset"
RELIABILITY_ID = "pashas/insurance-ai-reliability-benchmark"

bitext_ds = load_dataset(BITEXT_ID, split="train")

# Fallback loading strategy for the reliability benchmark dataset in case standard load_dataset fails
try:
    reliability_ds = load_dataset(RELIABILITY_ID, split="train")
except Exception:
    print("Standard load failed. Inspecting repository files...")
    files = list_repo_files(repo_id=RELIABILITY_ID, repo_type="dataset")
    print("Files found in repo:", files)

    # Try to find any training data file (json, jsonl, csv, or parquet)
    train_file = next((f for f in files if "train" in f.lower() and (f.endswith(".json") or f.endswith(".jsonl") or f.endswith(".parquet") or f.endswith(".csv"))), None)
    if not train_file:
        # Fallback to the first data-looking file if no explicit "train" is in the name
        train_file = next((f for f in files if f.endswith(".json") or f.endswith(".jsonl") or f.endswith(".parquet")), None)

    if not train_file:
        raise RuntimeError("Could not locate a supported dataset file in the repository.")

    print(f"Downloading {train_file} from Hugging Face hub...")
    local_file = hf_hub_download(repo_id=RELIABILITY_ID, repo_type="dataset", filename=train_file)

    if train_file.endswith(".parquet"):
        df = pd.read_parquet(local_file)
        reliability_ds = Dataset.from_pandas(df)
    elif train_file.endswith(".jsonl"):
        reliability_ds = load_dataset("json", data_files=local_file, split="train")
    elif train_file.endswith(".csv"):
        df = pd.read_csv(local_file)
        reliability_ds = Dataset.from_pandas(df)
    else:
        with open(local_file, "r", encoding="utf-8") as f:
            raw_data = json.load(f)
        reliability_ds = Dataset.from_list(raw_data)

BITEXT_FIELDS = ["instruction", "intent", "category", "tags", "response"]
missing_bitext = [name for name in BITEXT_FIELDS if name not in bitext_ds.column_names]
if missing_bitext:
    raise ValueError(f"Bitext columns changed; missing {missing_bitext}. Found: {bitext_ds.column_names}")

bitext_rows = []
for row in bitext_ds:
    clean = {name: str(row.get(name, "") or "").strip() for name in BITEXT_FIELDS}
    if all(clean.values()):
        bitext_rows.append(clean)

reliability_rows = [dict(row) for row in reliability_ds]
required_reliability = {"category", "subcategory", "difficulty", "input", "expected_output", "evaluation_criteria", "metadata"}
if not reliability_rows:
    raise RuntimeError("The reliability benchmark train split loaded empty.")
if not required_reliability.issubset(reliability_rows[0]):
    raise ValueError(f"Reliability schema changed. Found keys: {list(reliability_rows[0])}")

print("Bitext clean train rows:", len(bitext_rows))
print("Reliability benchmark train rows:", len(reliability_rows))
print("Bitext source columns:", bitext_ds.column_names)
print("Reliability example keys:", list(reliability_rows[0].keys()))

Standard load failed. Inspecting repository files...
Files found in repo: ['.gitattributes', 'README.md', 'data/benchmark_v1.jsonl', 'data/sample_predictions.jsonl', 'data/sample_results.json', 'data/test.jsonl', 'data/train.jsonl', 'data/validation.jsonl', 'evaluation/evaluate.py', 'evaluation/example_evaluation.py', 'generate_benchmark.py']
Bitext clean train rows: 39000
Reliability benchmark train rows: 357
Bitext source columns: ['instruction', 'intent', 'category', 'tags', 'response']
Reliability example keys: ['id', 'category', 'subcategory', 'difficulty', 'input', 'expected_output', 'evaluation_criteria', 'metadata']


## 6. Normalize the three tasks into one training mixture

In [12]:
# Per-source sample sizes. The benchmark is small, so it is mildly oversampled.
N_QA = 1500
N_BITEXT = 1500
N_RELIABILITY = 500

rng = random.Random(SEED)
if len(insuranceqa_rows) < N_QA:
    raise ValueError(f"Need {N_QA} InsuranceQA pairs but only found {len(insuranceqa_rows)}.")
if len(bitext_rows) < N_BITEXT:
    raise ValueError(f"Need {N_BITEXT} Bitext rows but only found {len(bitext_rows)}.")

qa_train_sample = rng.sample(insuranceqa_rows, N_QA)
bitext_train_sample = rng.sample(bitext_rows, N_BITEXT)
# With replacement only for this small TRAIN split. Validation and test remain untouched.
reliability_train_sample = rng.choices(reliability_rows, k=N_RELIABILITY)


def qa_instruction():
    return (
        "Generate one novel synthetic InsuranceQA-style question and professional answer. "
        "Return exactly one JSON object with keys domain, question, answer. "
        "Use the requested insurance domain. Do not copy any training question verbatim. "
        "This is synthetic text, not individualized insurance advice."
    )


def bitext_instruction():
    return (
        "Generate one novel synthetic insurance customer-support record. "
        "Return exactly one JSON object with keys instruction, intent, category, tags, response. "
        "Keep the requested intent, category, and tags. Do not copy a training utterance verbatim. "
        "Use placeholders rather than real personal, policy, claim, or payment data."
    )


def reliability_instruction():
    return (
        "Generate one novel synthetic insurance-agent benchmark scenario, not an answer to a real customer. "
        "Return exactly one JSON object with keys id, category, subcategory, difficulty, input, expected_output, "
        "evaluation_criteria, metadata. Keep the requested category, subcategory, difficulty, routing decision, "
        "and insurance line. Make the scenario internally consistent and include safe escalation when appropriate. "
        "Do not copy a training scenario or include real PII."
    )


def to_conversation(task, record):
    instructions = {
        "insuranceqa_v2": qa_instruction(),
        "bitext": bitext_instruction(),
        "reliability": reliability_instruction(),
    }
    user_text = instructions[task]
    return [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": json.dumps(record, ensure_ascii=False)},
    ]


training_records = []
for row in qa_train_sample:
    training_records.append(("insuranceqa_v2", row))
for row in bitext_train_sample:
    training_records.append(("bitext", row))
for row in reliability_train_sample:
    # Store exactly the benchmark schema; preserve nested fields.
    training_records.append(("reliability", row))
rng.shuffle(training_records)

print("Combined sample counts:", Counter(task for task, _ in training_records))
print("Total combined training examples:", len(training_records))

Combined sample counts: Counter({'insuranceqa_v2': 1500, 'bitext': 1500, 'reliability': 500})
Total combined training examples: 3500


## 7. Load Gemma 4 E2B Instruct in 4-bit mode and add text-only LoRA

In [13]:
from unsloth import FastModel

MODEL_NAME = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 1536

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    dtype=None,             # Let Unsloth choose a supported compute dtype for the Colab GPU.
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,      # QLoRA keeps VRAM use lower than the supplied notebook's 16-bit load.
    full_finetuning=False,
)

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    random_state=SEED,
)

print("Loaded:", MODEL_NAME)
print("4-bit QLoRA; text examples only.")

==((====))==  Unsloth 2026.10.11: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


model.safetensors.index.json:   0%|          | 0.00/301k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

Loaded: unsloth/gemma-4-E2B-it
4-bit QLoRA; text examples only.


## 8. Render examples with Gemma’s chat template

In [14]:
def format_as_text(examples):
    texts = []
    for task, record in examples:
        conversation = to_conversation(task, record)
        rendered = tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False,
        )
        # Gemma adds BOS from the tokenizer during tokenization; remove a literal
        # leading BOS here as recommended by the current Unsloth Gemma 4 guide.
        if tokenizer.bos_token:
            rendered = rendered.removeprefix(tokenizer.bos_token)
        texts.append(rendered)
    return texts

training_dataset = Dataset.from_dict({"text": format_as_text(training_records)})
print("Prepared rows:", len(training_dataset))
print(training_dataset[0]["text"][:1000])

Prepared rows: 3500
<|turn>user
Generate one novel synthetic InsuranceQA-style question and professional answer. Return exactly one JSON object with keys domain, question, answer. Use the requested insurance domain. Do not copy any training question verbatim. This is synthetic text, not individualized insurance advice.<turn|>
<|turn>model
{"domain": "health-insurance", "question": "How Does It Work If You Have Two Health Insurance Policies?", "answer": "For someone covered by two or more group health insurance policies, the term COB, short for coordination of benefits, would be used in such a scenario. Group insurance was designed to cover major medical expenses; however, it was never intended to pay in excess of 100% of incurred charges. For this reason,coordination of benefitswas established as a method by which two or more carriers or plans could coordinate their respective benefits so the total benefit paid does not exceed 100% of the total allowable expenses incurred. In most stat

## 9. Train one combined SFT model

In [15]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=training_dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        warmup_steps=10,
        learning_rate=2e-4,
        logging_steps=20,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        report_to="none",
    ),
)

# Only assistant completions are used for the loss, following the supplied notebook.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

# Basic sanity check: user prompt labels should be masked (-100), with assistant labels retained.
example = trainer.train_dataset[0]
print("Tokenized training example length:", len(example["input_ids"]))
print("Assistant labels retained:", sum(label != -100 for label in example["labels"]))

train_result = trainer.train()
print(train_result)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3500 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

Tokenized training example length: 280
Assistant labels retained: 222


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,500 | Num Epochs = 1 | Total steps = 875
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 12,668,928 of 5,135,846,944 (0.25% trained)


Step,Training Loss
20,2.869917
40,1.694805
60,1.391008
80,1.331123
100,1.142045
120,1.120406
140,0.918305
160,0.824127
180,0.869806
200,1.015844


TrainOutput(global_step=875, training_loss=1.0207077691214426, metrics={'train_runtime': 3154.6118, 'train_samples_per_second': 1.109, 'train_steps_per_second': 0.277, 'total_flos': 1.500803949623712e+16, 'train_loss': 1.0207077691214426, 'epoch': 1.0})


## 10. Save the LoRA adapter and tokenizer to Google Drive

In [16]:
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "gemma4_e2b_insurance_lora")
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved LoRA adapter and tokenizer:", ADAPTER_DIR)
print("This folder contains the adapter, not a merged standalone copy of the full base model.")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/gemma_insurance_synthetic/gemma4_e2b_insurance_lora/tokenizer_config.json.


Saved LoRA adapter and tokenizer: /content/drive/MyDrive/gemma_insurance_synthetic/gemma4_e2b_insurance_lora
This folder contains the adapter, not a merged standalone copy of the full base model.


## 11. Configure a mixed synthetic-data request

In [17]:
# Make a mixed output file. Edit N_SYNTHETIC_RECORDS and weights as needed.
N_SYNTHETIC_RECORDS = 30
TASK_WEIGHTS = {
    "insuranceqa_v2": 0.40,
    "bitext": 0.40,
    "reliability": 0.20,
}
TEMPERATURE = 0.8
TOP_P = 0.95
TOP_K = 64
MAX_NEW_TOKENS = 768
MAX_JSON_ATTEMPTS = 3

assert N_SYNTHETIC_RECORDS > 0
assert all(weight >= 0 for weight in TASK_WEIGHTS.values())
assert sum(TASK_WEIGHTS.values()) > 0

# Use the full TRAIN-only records for profile sampling, not validation/test.
def reliability_profile(row):
    expected = row.get("expected_output", {}) or {}
    metadata = row.get("metadata", {}) or {}
    return {
        "category": row.get("category", ""),
        "subcategory": row.get("subcategory", ""),
        "difficulty": row.get("difficulty", ""),
        "routing_decision": expected.get("routing_decision", ""),
        "insurance_line": metadata.get("insurance_line", ""),
    }


def sample_profile(task):
    if task == "insuranceqa_v2":
        row = rng.choice(insuranceqa_rows)
        return {"domain": row["domain"]}
    if task == "bitext":
        row = rng.choice(bitext_rows)
        return {"intent": row["intent"], "category": row["category"], "tags": row["tags"]}
    row = rng.choice(reliability_rows)
    return reliability_profile(row)


def allocate_exact_counts(total, weights):
    names = list(weights)
    weight_sum = sum(weights.values())
    raw = {name: total * weights[name] / weight_sum for name in names}
    counts = {name: math.floor(raw[name]) for name in names}
    left = total - sum(counts.values())
    order = sorted(names, key=lambda name: raw[name] - counts[name], reverse=True)
    for name in order[:left]:
        counts[name] += 1
    return counts


requested_counts = allocate_exact_counts(N_SYNTHETIC_RECORDS, TASK_WEIGHTS)
tasks_to_generate = []
for task, count in requested_counts.items():
    tasks_to_generate.extend([task] * count)
rng.shuffle(tasks_to_generate)
print("Requested generated task counts:", requested_counts)

Requested generated task counts: {'insuranceqa_v2': 12, 'bitext': 12, 'reliability': 6}


## 12. Generate records, enforce the sampled strata, and validate JSON

In [19]:
EXPECTED_KEYS = {
    "insuranceqa_v2": {"domain", "question", "answer"},
    "bitext": {"instruction", "intent", "category", "tags", "response"},
    "reliability": {"id", "category", "subcategory", "difficulty", "input", "expected_output", "evaluation_criteria", "metadata"},
}


def build_generation_prompt(task, profile):
    if task == "insuranceqa_v2":
        schema = {"domain": "string", "question": "string", "answer": "string"}
        instruction = qa_instruction()
    elif task == "bitext":
        schema = {"instruction": "string", "intent": "string", "category": "string", "tags": "string", "response": "string"}
        instruction = bitext_instruction()
    else:
        schema = {
            "id": "string",
            "category": "string",
            "subcategory": "string",
            "difficulty": "easy|medium|hard",
            "input": {"customer_message": "string", "context": "object", "conversation_history": "array"},
            "expected_output": {"intent": "string", "routing_decision": "ai_handle|ai_with_verification|human_handoff|hybrid_collaborative", "required_actions": "array", "priority": "string", "compliance_flags": "array", "expected_response_elements": "array"},
            "evaluation_criteria": "object with the benchmark-style scoring fields",
            "metadata": "object with insurance_line and relevant flags",
        }
        instruction = reliability_instruction()
    return (
        f"{instruction}\n\n"
        f"Use this sampled empirical profile exactly for the listed fields: {json.dumps(profile, ensure_ascii=False)}\n"
        f"Required JSON shape: {json.dumps(schema, ensure_ascii=False)}\n"
        "Return only one valid JSON object. Invent new customer wording and details. Use fictional IDs or placeholders; never include real personal data."
    )


def parse_first_json_object(text):
    start = text.find("{")
    if start < 0:
        raise ValueError("No JSON object found in model output")
    obj, _ = json.JSONDecoder().raw_decode(text[start:])
    if not isinstance(obj, dict):
        raise ValueError("Generated JSON value is not an object")
    return obj


def apply_profile(task, record, profile, index):
    # Constrain distribution-critical categorical fields to an observed joint profile.
    if task == "insuranceqa_v2":
        record["domain"] = profile["domain"]
    elif task == "bitext":
        for key in ("intent", "category", "tags"):
            record[key] = profile[key]
    else:
        record["id"] = f"SYNTHETIC_{index:05d}"
        for key in ("category", "subcategory", "difficulty"):
            record[key] = profile[key]
        record["expected_output"]["routing_decision"] = profile["routing_decision"]
        record["metadata"]["insurance_line"] = profile["insurance_line"]
    return record


def validate_generated(task, record):
    missing = EXPECTED_KEYS[task] - set(record)
    if missing:
        raise ValueError(f"Missing required keys: {sorted(missing)}")
    if task == "insuranceqa_v2":
        if not all(isinstance(record.get(key), str) and record[key].strip() for key in ("domain", "question", "answer")):
            raise ValueError("InsuranceQA fields must be non-empty strings")
    elif task == "bitext":
        if not all(isinstance(record.get(key), str) and record[key].strip() for key in EXPECTED_KEYS[task]):
            raise ValueError("Bitext fields must be non-empty strings")
    else:
        for key in ("input", "expected_output", "evaluation_criteria", "metadata"):
            if not isinstance(record.get(key), dict):
                raise ValueError(f"Reliability field {key} must be a JSON object")
        if not isinstance(record["expected_output"].get("required_actions"), list):
            raise ValueError("Reliability required_actions must be a JSON array")
    return True


def generate_one(task, profile, index):
    prompt = build_generation_prompt(task, profile)
    # Wrap prompt explicitly as text elements to bypass Hugging Face's 5.5.0 multi-modal checks
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")
    input_length = inputs["input_ids"].shape[-1]
    for attempt in range(1, MAX_JSON_ATTEMPTS + 1):
        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                top_k=TOP_K,
                use_cache=True,  # Important for Gemma 4 E2B generation.
                pad_token_id=tokenizer.eos_token_id,
            )
        generated_text = tokenizer.decode(output[0][input_length:], skip_special_tokens=True)
        try:
            record = parse_first_json_object(generated_text)
            if task == "reliability":
                for nested in ("expected_output", "metadata"):
                    if not isinstance(record.get(nested), dict):
                        raise ValueError(f"Reliability field {nested} must be an object before profile enforcement")
            record = apply_profile(task, record, profile, index)
            validate_generated(task, record)
            return record
        except Exception as exc:
            if attempt == MAX_JSON_ATTEMPTS:
                print(f"Skipping record {index} ({task}) after {attempt} attempts: {exc}")
            # Retry the same sampled profile, retaining the requested distribution.
    return None


generated_records = []
for index, task in enumerate(tasks_to_generate, start=1):
    profile = sample_profile(task)
    record = generate_one(task, profile, index)
    if record is not None:
        generated_records.append({"dataset_type": task, "record": record})
    if index % 5 == 0 or index == len(tasks_to_generate):
        print(f"Processed {index}/{len(tasks_to_generate)}; valid records: {len(generated_records)}")

print("Valid generated records:", len(generated_records))

Processed 5/30; valid records: 5
Processed 10/30; valid records: 10
Processed 15/30; valid records: 15
Processed 20/30; valid records: 20
Processed 25/30; valid records: 25
Processed 30/30; valid records: 30
Valid generated records: 30


## 13. Export JSONL and CSV, then compare key distributions

In [20]:
if not generated_records:
    raise RuntimeError("No valid records were generated. Check GPU memory, model output, and the JSON validation messages.")

jsonl_path = os.path.join(OUTPUT_DIR, "synthetic_insurance_records.jsonl")
csv_path = os.path.join(OUTPUT_DIR, "synthetic_insurance_records.csv")

with open(jsonl_path, "w", encoding="utf-8") as handle:
    for item in generated_records:
        handle.write(json.dumps(item, ensure_ascii=False) + "\n")

# Mixed schemas are represented in CSV as dataset_type plus one JSON-encoded record column.
# This is lossless and keeps nested benchmark fields intact.
csv_rows = [
    {"dataset_type": item["dataset_type"], "record_json": json.dumps(item["record"], ensure_ascii=False)}
    for item in generated_records
]
pd.DataFrame(csv_rows).to_csv(csv_path, index=False, encoding="utf-8")

print("Saved JSONL:", jsonl_path)
print("Saved CSV:", csv_path)
print("Generated by task:", Counter(item["dataset_type"] for item in generated_records))


def key_distribution(items, task, getter):
    values = []
    for item in items:
        if item["dataset_type"] == task:
            value = getter(item["record"])
            if value not in (None, ""):
                values.append(str(value))
    return Counter(values)


# Show empirical reference vs generated distributions for the most relevant labels.
qa_reference = [{"dataset_type": "insuranceqa_v2", "record": row} for row in insuranceqa_rows]
bitext_reference = [{"dataset_type": "bitext", "record": row} for row in bitext_rows]
reliability_reference = [{"dataset_type": "reliability", "record": row} for row in reliability_rows]

comparisons = [
    ("InsuranceQA domain", "insuranceqa_v2", lambda record: record.get("domain")),
    ("Bitext intent", "bitext", lambda record: record.get("intent")),
    ("Bitext category", "bitext", lambda record: record.get("category")),
    ("Reliability category", "reliability", lambda record: record.get("category")),
    ("Reliability difficulty", "reliability", lambda record: record.get("difficulty")),
    ("Reliability routing", "reliability", lambda record: (record.get("expected_output") or {}).get("routing_decision")),
]
reference_groups = {
    "insuranceqa_v2": qa_reference,
    "bitext": bitext_reference,
    "reliability": reliability_reference,
}
for label, task, getter in comparisons:
    reference = key_distribution(reference_groups[task], task, getter)
    generated = key_distribution(generated_records, task, getter)
    combined = sorted(set(reference) | set(generated), key=lambda value: (-(reference[value]), value))[:12]
    table = pd.DataFrame({
        "label": combined,
        "train_count": [reference[value] for value in combined],
        "train_share": [round(reference[value] / max(1, sum(reference.values())), 3) for value in combined],
        "generated_count": [generated[value] for value in combined],
        "generated_share": [round(generated[value] / max(1, sum(generated.values())), 3) for value in combined],
    })
    print("\n" + label)
    display(table)

print("\nSample output:")
print(json.dumps(generated_records[0], ensure_ascii=False, indent=2)[:2500])

Saved JSONL: /content/drive/MyDrive/gemma_insurance_synthetic/synthetic_insurance_records.jsonl
Saved CSV: /content/drive/MyDrive/gemma_insurance_synthetic/synthetic_insurance_records.csv
Generated by task: Counter({'insuranceqa_v2': 12, 'bitext': 12, 'reliability': 6})

InsuranceQA domain


,label,train_count,train_share,generated_count,generated_share
0,life-insurance,3328,0.258,2,0.167
1,medicare-insurance,1770,0.137,1,0.083
2,auto-insurance,1659,0.129,0,0.000
3,health-insurance,1542,0.120,0,0.000
4,home-insurance,1340,0.104,4,0.333
5,renters-insurance,800,0.062,2,0.167
6,disability-insurance,751,0.058,1,0.083
7,annuities,611,0.047,0,0.000
8,retirement-plans,525,0.041,2,0.167
9,long-term-care-insurance,427,0.033,0,0.000



Bitext intent


,label,train_count,train_share,generated_count,generated_share
0,accept_settlement,1000,0.026,1,0.083
1,agent,1000,0.026,1,0.083
2,appeal_denied_insurance_claim,1000,0.026,0,0.000
3,buy_insurance_policy,1000,0.026,0,0.000
4,calculate_insurance_quote,1000,0.026,0,0.000
5,cancel_insurance_policy,1000,0.026,1,0.083
6,cancellation_fees,1000,0.026,1,0.083
7,change_coverage,1000,0.026,1,0.083
8,change_personal_details,1000,0.026,0,0.000
9,check_coverage,1000,0.026,0,0.000



Bitext category


,label,train_count,train_share,generated_count,generated_share
0,CLAIMS,6000,0.154,2,0.167
1,PAYMENT,5000,0.128,0,0.000
2,CONTACT,4000,0.103,4,0.333
3,COVERAGE,4000,0.103,1,0.083
4,ENROLLMENT,4000,0.103,2,0.167
5,COMPLAINTS,3000,0.077,0,0.000
6,INCIDENTS,2000,0.051,0,0.000
7,QUOTE,2000,0.051,1,0.083
8,AUTO_INSURANCE,1000,0.026,0,0.000
9,GENERAL_INFORMATION,1000,0.026,0,0.000


... (3 outputs left out)

,label,train_count,train_share,generated_count,generated_share
0,medium,162,0.454,2,0.333
1,hard,100,0.280,1,0.167
2,easy,95,0.266,3,0.500



Reliability routing


,label,train_count,train_share,generated_count,generated_share
0,ai_handle,141,0.395,5,0.833
1,human_handoff,102,0.286,1,0.167
2,ai_with_verification,98,0.275,0,0.000
3,hybrid_collaborative,16,0.045,0,0.000



Sample output:
{
  "dataset_type": "insuranceqa_v2",
  "record": {
    "domain": "home-insurance",
    "question": "How Does Homeowners Insurance Work For Newly Built Homes?",
    "answer": "Homeowners insurance is designed to cover the structure and contents of your new home and provide coverage for personal liability. The type of policy you select will determine the level of coverage provided. The structure of your home, along with the coverage you select, will determine the policy limits. For example, if your home is constructed of concrete and wood framing, it will have a different level of coverage and a different premium than if it were built with cinder block and plaster. The deductible you pay for a claim will also vary based on the coverage you choose. It is wise to consult with an experienced insurance broker to help you select a policy that meets your needs and budget."
  }
}


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import torch
import gc
import sys
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

# Completely purge references to free GPU memory
for var in ["model", "tokenizer", "trainer"]:
    if var in globals():
        del globals()[var]

# Clear IPython output histories to release cached tensors
if "ip" in globals() or "get_ipython" in globals():
    try:
        ipython = get_ipython()
        ipython.user_ns.pop("model", None)
        ipython.user_ns.pop("tokenizer", None)
        ipython.user_ns.pop("trainer", None)
        ipython.history_manager.reset(new_session=False)
    except Exception:
        pass

# Purge system-wide references and empty cache
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f"Free GPU Memory after clean-up: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")

ADAPTER_DIR = (
    "/content/drive/MyDrive/gemma_insurance_synthetic/"
    "gemma4_e2b_insurance_lora"
)

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU in Colab: Runtime > Change runtime type > GPU.")

# Reload the fine-tuned adapter from GDrive
model, tokenizer = FastModel.from_pretrained(
    model_name=ADAPTER_DIR,
    dtype=None,
    max_seq_length=1536,
    load_in_4bit=True,
    full_finetuning=False,
)

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
model.eval()

print("Loaded the fine-tuned adapter successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Free GPU Memory after clean-up: 14.41 GB
==((====))==  Unsloth 2026.10.11: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

Loaded the fine-tuned adapter successfully.


In [3]:
USER_PROMPT = """
Generate 5 synthetic records based strictly on the insurance datasets learned
during fine-tuning. Preserve their insurance terminology, placeholders,
linguistic variations, and response-writing patterns. Do not copy records
verbatim. Do not invent insurance facts. Use only labels and categories seen
in the training data. Output only valid JSON data.
"""

def generate_text(prompt, max_new_tokens=1800):
    # Wrap prompt in a rich-text block structure to bypass Transformers 5.5.0's multi-modal checks
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")

    prompt_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            top_k=64,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        output[0][prompt_length:],
        skip_special_tokens=True,
    ).strip()

answer = generate_text(USER_PROMPT)
print(answer)

{"policy_type": "auto", "policy_status": "active", "policy_status_date": "2023-10-15", "policy_type_details": {"coverage": "all_risk", "deductible_amount": "500", "limits": {"liability": "100/300/500", "collision": "500/250/1000", "comprehensive": "500", "other": "250"}, "status": "active", "effective_date": "2023-01-01", "expiration_date": "2024-01-01", "premium": 1200.00, "carrier": "BlueSky Insurance", "status_description": "Active coverage for named auto policy", "issue_year": 2023, "policy_number": "POL-23-902101", "status_history": ["active", "renewal_pending"]}, {"policy_type": "renters", "policy_status": "active", "policy_status_date": "2023-11-01", "policy_type_details": {"coverage": "personal_property", "deductible_amount": "1000", "limits": {"personal_property": "150000", "liability": "100000/300000/500000"}, "status": "active", "effective_date": "2023-11-01", "expiration_date": "2024-11-01", "premium": 750.00, "carrier": "BlueSky Insurance", "status_description": "Active co

## Notes

- The output is one **mixed** file. Every JSONL line contains `dataset_type` and a source-shaped `record`. The CSV has the same information in `dataset_type` and a lossless JSON string in `record_json`.
- Adjust `TASK_WEIGHTS` to change the proportion of the three output record types. Label/category proportions within each task are sampled from the corresponding training data and then enforced in the generated JSON.
- `MAX_SEQ_LENGTH`, record sample sizes, and generation count affect runtime and GPU memory. If the GPU runs out of memory, lower `MAX_SEQ_LENGTH` (for example to 1024), reduce sample counts, and restart the runtime before rerunning.
- Generated examples can still contain inaccurate insurance statements or malformed semantic details even when the JSON parses. Review and filter synthetic data before using it for downstream training.
- The saved directory contains a **LoRA adapter**, not a merged full model. To continue using it in a fresh runtime, load the same base model first and then load the adapter with PEFT/Unsloth.

Sources and format references: [InsuranceQA V2 README](https://github.com/shuzi/insuranceqa/blob/master/V2/README.md), [Bitext dataset card](https://huggingface.co/datasets/bitext/Bitext-insurance-llm-chatbot-training-dataset), [reliability benchmark card](https://huggingface.co/datasets/pashas/insurance-ai-reliability-benchmark), [Unsloth Gemma 4 fine-tuning guide](https://unsloth.ai/docs/models/gemma-4/train).